# COMP0513 – Representação de Conhecimento e Raciocínio Automático
## Parte I – Unidade 01 (Aulas 01 a 10) | Grupo 7

**Integrantes:** 
- SAMUEL CAMPOS DOS SANTOS (202600081368)
- CAIO VICTOR PRADO CRUZ (202100011234)
- JOAO PEDRO CHAVES OLIVEIRA (202600046064)

**Domínio:** Biblioteca Universitária – controle de acervo e empréstimos (atraso de devolução e bloqueio para novos empréstimos).
**Motor de inferência:** `aima3` (aima-python) – `FolKB`, `expr`, `fol_fc_ask`, `fol_bc_ask`, `unify`, `subst`.

**Como executar no Google Colab:** `Ambiente de execução → Executar tudo` (Run all). A única dependência é a biblioteca `aima3`, instalada na célula abaixo.

**Roteiro deste notebook**
1. Domínio, propósito, escopo e pressupostos
2. Vocabulário e assinatura da KB
3. Base de conhecimento (fatos F1–F13 e regras R1–R4)
4. Consultas de competência PC1–PC12 (ground e com variáveis)
5. Unificação e MGU (com falha de unificação)
6. Encadeamento para frente (forward chaining) com rastro
7. Encadeamento para trás (backward chaining) com decomposição de metas
8. Sistema de produção com memória de trabalho (ciclo reconhecer–resolver conflito–agir)
9. Atualização dinâmica (TELL / RETRACT)
10. Testes automatizados de regressão
11. Limitações e fronteiras do modelo

In [1]:
!pip install aima3 -q

## 1. Domínio, propósito, escopo e pressupostos

**Propósito:** automatizar o raciocínio dedutivo sobre regras de empréstimo da biblioteca: associar usuário → empréstimo → livro, verificar prazos de devolução pendentes e **inferir** automaticamente o estado de atraso e o bloqueio para novos empréstimos — respostas que **não** estão armazenadas explicitamente (ex.: `BloqueadoParaEmprestimo(Ana)` é derivado por uma cadeia de 3 regras).

**Escopo incluído:** usuários (Ana, Bruno), livros (LivroA, LivroB), categorias (Computacao), registro de empréstimo (Emprestimo1), status (em aberto), datas (Data10, Data15) e as relações de posse de livro, vencimento de prazo, pendência e bloqueio.
**Escopo excluído:** multas financeiras, renovações, reservas, devolução parcial, funcionários e aquisição de acervo.

**Pressupostos explícitos**
- **P1 (Mundo Aberto):** a ausência de um fato na KB significa apenas "não derivável", e **não** falso no mundo real.
- **P2 (Temporalidade estática):** a ordem entre datas é dada por fatos explícitos de `Anterior(d1, d2)`; não há aritmética de calendário.
- **P3 (Data corrente fixa):** a data atual é o fato `DataAtual(Data15)`.
- **P4 (Estado do empréstimo):** o atraso só é considerado se o empréstimo estiver explicitamente `EmAberto`.
- **P5 (Garantia de categoria):** todo livro do acervo possui alguma categoria (regra teórica R5).

**Separação declarativo × procedural:** fatos e regras lógicas vivem na KB (`FolKB`) e são **declarativos**; as regras de produção da Seção 8 são **procedurais** e agem sobre uma memória de trabalho dinâmica, separada da KB.

## 2. Vocabulário e assinatura da KB

Constantes: `Ana`, `Bruno`, `LivroA`, `LivroB`, `Computacao`, `Emprestimo1`, `Data10` (10/09/2026), `Data15` (15/09/2026).

| Predicado | Aridade | Argumentos (ordem fixa) | Significado |
|---|---|---|---|
| `Usuario(u)` | 1 | indivíduo | u é usuário cadastrado |
| `Livro(l)` | 1 | objeto | l é livro do acervo |
| `Categoria(c)` | 1 | conceito | c é categoria de conhecimento |
| `Emprestimo(e)` | 1 | registro | e é registro de empréstimo |
| `EmAberto(e)` | 1 | registro | o empréstimo e não foi devolvido |
| `CategoriaDoLivro(l, c)` | 2 | livro, categoria | l pertence à categoria c |
| `TemEmprestimo(u, e)` | 2 | usuário, empréstimo | u é responsável por e |
| `LivroDoEmprestimo(e, l)` | 2 | empréstimo, livro | e refere-se ao livro l |
| `PegouEmprestado(u, l)` | 2 | usuário, livro | **derivado (R1)** – u retirou l |
| `DataDevolucao(e, d)` | 2 | empréstimo, data | prazo de devolução de e é d |
| `DataAtual(d)` | 1 | data | data corrente do sistema |
| `Anterior(d1, d2)` | 2 | data, data | d1 ocorreu antes de d2 |
| `EmprestimoAtrasado(e)` | 1 | empréstimo | **derivado (R2)** – e vencido e não devolvido |
| `Atrasado(u)` | 1 | usuário | **derivado (R3)** – u com pendência de atraso |
| `BloqueadoParaEmprestimo(u)` | 1 | usuário | **derivado (R4)** – u impedido de novos empréstimos |

## 3. Base de conhecimento em `FolKB` (fatos + regras)

Os fatos F1–F13 e as regras R1–R4 são declarados em uma única lista auditável e carregados no `FolKB`.
A função `construir_kb()` reconstrói a KB sempre que necessário (por exemplo, para isolar seções que alteram o estado, como o encadeamento para frente e o `retract`).

In [2]:
from aima3.logic import FolKB, expr, fol_fc_ask, fol_bc_ask, unify, subst

CLAUSULAS = [
    # ---------- FATOS EXPLICITOS (F1-F13) ----------
    expr('Usuario(Ana)'),                              # F1
    expr('Usuario(Bruno)'),                            # F2
    expr('Livro(LivroA)'),                             # F3
    expr('Livro(LivroB)'),                             # F4
    expr('Categoria(Computacao)'),                     # F5
    expr('CategoriaDoLivro(LivroA, Computacao)'),      # F6
    expr('Emprestimo(Emprestimo1)'),                   # F7
    expr('TemEmprestimo(Ana, Emprestimo1)'),           # F8
    expr('LivroDoEmprestimo(Emprestimo1, LivroA)'),    # F9
    expr('EmAberto(Emprestimo1)'),                     # F10
    expr('DataDevolucao(Emprestimo1, Data10)'),        # F11
    expr('DataAtual(Data15)'),                         # F12
    expr('Anterior(Data10, Data15)'),                  # F13
    # ---------- REGRAS LOGICAS GERAIS (R1-R4) ----------
    # R1: posse indireta de livro
    expr('(TemEmprestimo(u, e) & LivroDoEmprestimo(e, l)) ==> PegouEmprestado(u, l)'),
    # R2: identificacao de emprestimo atrasado
    expr('(Emprestimo(e) & EmAberto(e) & DataDevolucao(e, d1) & DataAtual(d2) & Anterior(d1, d2)) ==> EmprestimoAtrasado(e)'),
    # R3: identificacao de usuario em atraso
    expr('(Usuario(u) & TemEmprestimo(u, e) & EmprestimoAtrasado(e)) ==> Atrasado(u)'),
    # R4: bloqueio para novos emprestimos
    expr('Atrasado(u) ==> BloqueadoParaEmprestimo(u)'),
]

def construir_kb():
    """(Re)constroi a KB a partir da lista fixa de clausulas (fatos + regras R1-R4)."""
    return FolKB(CLAUSULAS)

kb = construir_kb()
print('Clausulas carregadas na KB:', len(kb.clauses))
for c in kb.clauses:
    print(' -', c)

Clausulas carregadas na KB: 17
 - Usuario(Ana)
 - Usuario(Bruno)
 - Livro(LivroA)
 - Livro(LivroB)
 - Categoria(Computacao)
 - CategoriaDoLivro(LivroA, Computacao)
 - Emprestimo(Emprestimo1)
 - TemEmprestimo(Ana, Emprestimo1)
 - LivroDoEmprestimo(Emprestimo1, LivroA)
 - EmAberto(Emprestimo1)
 - DataDevolucao(Emprestimo1, Data10)
 - DataAtual(Data15)
 - Anterior(Data10, Data15)
 - ((TemEmprestimo(u, e) & LivroDoEmprestimo(e, l)) ==> PegouEmprestado(u, l))
 - (((((Emprestimo(e) & EmAberto(e)) & DataDevolucao(e, d1)) & DataAtual(d2)) & Anterior(d1, d2)) ==> EmprestimoAtrasado(e))
 - (((Usuario(u) & TemEmprestimo(u, e)) & EmprestimoAtrasado(e)) ==> Atrasado(u))
 - (Atrasado(u) ==> BloqueadoParaEmprestimo(u))


In [3]:
def limpar(solucoes):
    """Exibe apenas as variaveis nomeadas da consulta (esconde variaveis auxiliares internas v_*)."""
    return [{str(k): v for k, v in s.items() if not str(k).startswith('v_')} for s in solucoes]

def provado(solucoes):
    """Consulta ground: ha pelo menos uma prova?"""
    return 'Verdadeiro (provado)' if list(solucoes) else 'Falso (nao provado)'

def valor(sub, nome):
    """Le o valor de uma variavel (por nome) em uma substituicao do aima3, como texto legivel."""
    for k, v in sub.items():
        if str(k) == nome:
            return str(v)  # termos do aima3 sao Expr; str() devolve o nome legivel da constante
    return None

## 4. Consultas de competência PC1–PC12

12 perguntas de competência, cobrindo: recuperação fechada (fato explícito), recuperação aberta (varável), inferência em cadeia, consulta existencial e caso **não provado** (indeterminação — pressuposto P1).

Observações:
- PC10 e PC12 exemplificam que "não provado" **não** significa "falso" (mundo aberto).
- PC11/PC12 envolvem a regra teórica R5 (`∀l Livro(l) → ∃c CategoriaDoLivro(l, c)`), que **não** é codificada no `FolKB` porque o motor SLD aceita apenas cláusulas definidas puras (sem quantificador existencial no consequente) — limitação declarada no item 11.

In [4]:
kb = construir_kb()  # KB limpa para as consultas

print('PC1  Ana e usuaria?                          ->', provado(fol_bc_ask(kb, expr('Usuario(Ana)'))))
print('PC2  Data de devolucao de Emprestimo1        ->', limpar(fol_bc_ask(kb, expr('DataDevolucao(Emprestimo1, d)'))))
print('PC3  Categoria de LivroA                     ->', limpar(fol_bc_ask(kb, expr('CategoriaDoLivro(LivroA, c)'))))
print('PC4  Quem pegou LivroA? (inf. R1)            ->', limpar(fol_bc_ask(kb, expr('PegouEmprestado(x, LivroA)'))))
print('PC5  Emprestimo1 esta atrasado? (inf. R2)    ->', provado(fol_bc_ask(kb, expr('EmprestimoAtrasado(Emprestimo1)'))))
print('PC6  Ana esta em atraso? (cadeia R2+R3)      ->', provado(fol_bc_ask(kb, expr('Atrasado(Ana)'))))
print('PC7  Ana esta bloqueada? (cadeia R2+R3+R4)   ->', provado(fol_bc_ask(kb, expr('BloqueadoParaEmprestimo(Ana)'))))
print('PC8  Quais usuarios estao bloqueados?        ->', limpar(fol_bc_ask(kb, expr('BloqueadoParaEmprestimo(x)'))))
print('PC9  Existe algum usuario bloqueado?         ->', provado(fol_bc_ask(kb, expr('BloqueadoParaEmprestimo(x)'))))
print('PC10 Bruno esta bloqueado?                   ->', limpar(fol_bc_ask(kb, expr('BloqueadoParaEmprestimo(Bruno)'))), '  <- lista vazia: nao derivavel (P1)')
print('PC11/PC12 LivroB tem categoria? (R5 teorica) ->', limpar(fol_bc_ask(kb, expr('CategoriaDoLivro(LivroB, c)'))), '  <- R5 fora do FolKB: ver nota')

PC1  Ana e usuaria?                          -> Verdadeiro (provado)
PC2  Data de devolucao de Emprestimo1        -> [{'d': Data10}]
PC3  Categoria de LivroA                     -> [{'c': Computacao}]
PC4  Quem pegou LivroA? (inf. R1)            -> [{'x': Ana}]
PC5  Emprestimo1 esta atrasado? (inf. R2)    -> Verdadeiro (provado)
PC6  Ana esta em atraso? (cadeia R2+R3)      -> Verdadeiro (provado)
PC7  Ana esta bloqueada? (cadeia R2+R3+R4)   -> Verdadeiro (provado)
PC8  Quais usuarios estao bloqueados?        -> [{'x': Ana}]
PC9  Existe algum usuario bloqueado?         -> Verdadeiro (provado)
PC10 Bruno esta bloqueado?                   -> []   <- lista vazia: nao derivavel (P1)
PC11/PC12 LivroB tem categoria? (R5 teorica) -> []   <- R5 fora do FolKB: ver nota


## 5. Unificação, substituições e MGU

A unificação encontra a substituição θ (mais geral possível, MGU) que torna duas expressões idênticas — é o coração da resolução e do encadeamento. A célula mostra:

1. **MGU** entre a meta `BloqueadoParaEmprestimo(Ana)` e a cabeça da regra R4 (`BloqueadoParaEmprestimo(u)`): θ = {u/Ana}.
2. **Aplicação da substituição** à regra R4 instanciada.
3. MGU com **duas variáveis**.
4. **Falha de unificação** por constantes distintas e por predicados distintos (retorna `None`).

In [5]:
# 1) MGU entre meta e cabeca da R4
mgu = unify(expr('BloqueadoParaEmprestimo(Ana)'), expr('BloqueadoParaEmprestimo(u)'))
print('MGU meta x cabeca R4:', mgu)

# 2) Substituicao aplicada a regra R4
print('R4 instanciada por theta:', subst(mgu, expr('Atrasado(u) ==> BloqueadoParaEmprestimo(u)')))

# 3) MGU com duas variaveis
mgu2 = unify(expr('TemEmprestimo(u, e)'), expr('TemEmprestimo(Ana, Emprestimo1)'))
print('MGU 2 variaveis:', mgu2)

# 4) Falhas de unificacao
print('Falha (constantes distintas Usuario(Ana) x Usuario(Bruno)):', unify(expr('Usuario(Ana)'), expr('Usuario(Bruno)')))
print('Falha (predicados distintos Usuario(Ana) x Livro(Ana)):', unify(expr('Usuario(Ana)'), expr('Livro(Ana)')))

MGU meta x cabeca R4: {u: Ana}
R4 instanciada por theta: (Atrasado(Ana) ==> BloqueadoParaEmprestimo(Ana))
MGU 2 variaveis: {u: Ana, e: Emprestimo1}
Falha (constantes distintas Usuario(Ana) x Usuario(Bruno)): None
Falha (predicados distintos Usuario(Ana) x Livro(Ana)): None


## 6. Encadeamento para frente (forward chaining)

**Data-driven:** parte dos fatos conhecidos e dispara regras cujos antecedentes estão satisfeitos, materializando novas conclusões até a saturação.

- `fol_fc_ask` (biblioteca): prova consultas pelo fechamento para frente. Observação prática: nessa versão do `aima3`, os fatos derivados são **adicionados à KB** (efeito colateral observável — mostrado abaixo).
- O **mini motor com rastro** (código didático sobre a mesma KB) imprime, em cada passo: regra disparada, substituição θ e novo fato derivado — atendendo ao requisito de rastreabilidade do encadeamento.

In [6]:
# ---- 6a) fol_fc_ask em KB dedicada (para observar o efeito colateral) ----
kb_fc = construir_kb()
antes = len(kb_fc.clauses)

print('FC: PegouEmprestado(x, LivroA)      ->', limpar(fol_fc_ask(kb_fc, expr('PegouEmprestado(x, LivroA)'))))
print('FC: BloqueadoParaEmprestimo(Ana)    ->', 'provado' if list(fol_fc_ask(kb_fc, expr('BloqueadoParaEmprestimo(Ana)'))) else 'nao provado')
print('FC: BloqueadoParaEmprestimo(Bruno)  ->', 'provado' if list(fol_fc_ask(kb_fc, expr('BloqueadoParaEmprestimo(Bruno)'))) else 'nao provado (mundo aberto)')

novos = [c for c in kb_fc.clauses if not any(c == b for b in CLAUSULAS)]
print(f'\nClausulas antes: {antes} | depois do FC: {len(kb_fc.clauses)}')
print('Fatos derivados materializados pelo encadeamento para frente:')
for c in novos:
    print(' +', c)

FC: PegouEmprestado(x, LivroA)      -> [{'x': Ana}]
FC: BloqueadoParaEmprestimo(Ana)    -> provado
FC: BloqueadoParaEmprestimo(Bruno)  -> nao provado (mundo aberto)

Clausulas antes: 17 | depois do FC: 21
Fatos derivados materializados pelo encadeamento para frente:
 + PegouEmprestado(Ana, LivroA)
 + EmprestimoAtrasado(Emprestimo1)
 + Atrasado(Ana)
 + BloqueadoParaEmprestimo(Ana)


In [7]:
# ---- 6b) Mini motor de encadeamento para frente com rastro (didatico) ----
REGRAS_TRACE = [
    ('R1', expr('PegouEmprestado(u, l)'), [expr('TemEmprestimo(u, e)'), expr('LivroDoEmprestimo(e, l)')]),
    ('R2', expr('EmprestimoAtrasado(e)'), [expr('Emprestimo(e)'), expr('EmAberto(e)'), expr('DataDevolucao(e, d1)'), expr('DataAtual(d2)'), expr('Anterior(d1, d2)')]),
    ('R3', expr('Atrasado(u)'), [expr('Usuario(u)'), expr('TemEmprestimo(u, e)'), expr('EmprestimoAtrasado(e)')]),
    ('R4', expr('BloqueadoParaEmprestimo(u)'), [expr('Atrasado(u)')]),
]
FATOS_INICIAIS = CLAUSULAS[:13]  # F1-F13 nesta ordem

def satisfaz(literal, fatos, lig):
    """Todas as extensoes da ligacao atual que unificam o literal com algum fato."""
    out = []
    for f in fatos:
        lig2 = unify(literal, f, dict(lig))
        if lig2 is not None:
            out.append(lig2)
    return out

def fc_com_rastro(fatos, regras):
    """Saturacao para frente com registro (passo, regra, theta, novo fato)."""
    fatos, rastro, passo, mudou = set(fatos), [], 0, True
    while mudou:
        mudou = False
        for nome, cabeca, corpo in regras:
            ligs = [dict()]
            for lit in corpo:
                novas = []
                for lig in ligs:
                    novas.extend(satisfaz(lit, fatos, lig))
                ligs = novas
                if not ligs:
                    break
            for lig in ligs:
                novo = subst(lig, cabeca)
                if novo not in fatos:
                    passo += 1
                    fatos.add(novo)
                    mudou = True
                    rastro.append((passo, nome, lig, novo))
    return fatos, rastro

fatos_fc, rastro = fc_com_rastro(FATOS_INICIAIS, REGRAS_TRACE)
print('RASTRO DO ENCADEAMENTO PARA FRENTE:')
for passo, nome, lig, novo in rastro:
    theta = {str(k): v for k, v in lig.items() if not str(k).startswith('v_')}
    print(f'  passo {passo}: {nome} dispara com \u03b8={theta}  ->  novo fato: {novo}')

assert len(rastro) == 4
assert expr('BloqueadoParaEmprestimo(Ana)') in fatos_fc
print('\nTotal de fatos derivados ate a saturacao:', len(rastro), '(R1, R2, R3 e R4 ao longo das passadas)')

RASTRO DO ENCADEAMENTO PARA FRENTE:
  passo 1: R1 dispara com θ={'u': Ana, 'e': Emprestimo1, 'l': LivroA}  ->  novo fato: PegouEmprestado(Ana, LivroA)
  passo 2: R2 dispara com θ={'e': Emprestimo1, 'd1': Data10, 'd2': Data15}  ->  novo fato: EmprestimoAtrasado(Emprestimo1)
  passo 3: R3 dispara com θ={'u': Ana, 'e': Emprestimo1}  ->  novo fato: Atrasado(Ana)
  passo 4: R4 dispara com θ={'u': Ana}  ->  novo fato: BloqueadoParaEmprestimo(Ana)

Total de fatos derivados ate a saturacao: 4 (R1, R2, R3 e R4 ao longo das passadas)


## 7. Encadeamento para trás (backward chaining)

**Goal-driven:** parte da meta e a decompõe em submetas, usando as regras de trás para frente.

Árvore de decomposição da meta **PC7 `BloqueadoParaEmprestimo(Ana)`**:

```
BloqueadoParaEmprestimo(Ana)          [R4: Atrasado(Ana)]
└── Atrasado(Ana)                     [R3: Usuario(Ana) & TemEmprestimo(Ana, e) & EmprestimoAtrasado(e)]
    ├── Usuario(Ana)                  [F1]
    ├── TemEmprestimo(Ana, Emprestimo1) [F8]
    └── EmprestimoAtrasado(Emprestimo1) [R2: Emprestimo1 em aberto + prazos]
        ├── Emprestimo(Emprestimo1)   [F7]
        ├── EmAberto(Emprestimo1)     [F10]
        ├── DataDevolucao(Emprestimo1, Data10) [F11]
        ├── DataAtual(Data15)         [F12]
        └── Anterior(Data10, Data15)  [F13]
```
Para **Bruno**, a decomposição falha em `Atrasado(Bruno)` → `TemEmprestimo(Bruno, e)`: não há fato, logo a meta não é provada (mundo aberto).

In [8]:
kb_bc = construir_kb()
metas = [
    'BloqueadoParaEmprestimo(Ana)',
    'Atrasado(Ana)',
    'EmprestimoAtrasado(Emprestimo1)',
    'TemEmprestimo(Ana, Emprestimo1)',
    'BloqueadoParaEmprestimo(Bruno)',
]
print('Encadeamento para tras (submetas na ordem da prova):')
for m in metas:
    r = list(fol_bc_ask(kb_bc, expr(m)))
    print(f'  meta {m:35s} ->', 'PROVADA   ' + str(limpar(r)) if r else 'NAO PROVADA ([] )')

Encadeamento para tras (submetas na ordem da prova):
  meta BloqueadoParaEmprestimo(Ana)        -> PROVADA   [{}]
  meta Atrasado(Ana)                       -> PROVADA   [{}]
  meta EmprestimoAtrasado(Emprestimo1)     -> PROVADA   [{}]
  meta TemEmprestimo(Ana, Emprestimo1)     -> PROVADA   [{}]
  meta BloqueadoParaEmprestimo(Bruno)      -> NAO PROVADA ([] )


## 8. Sistema de produção com memória de trabalho (Aulas 09–10)

Camada **procedural**, separada da KB lógica: regras `IF condições THEN ações` que **alteram** uma **memória de trabalho (WM)** dinâmica — não derivam conclusões sobre o mundo, e sim executam ações administrativas do fluxo de atraso (registrar atraso, criar pendência, aplicar bloqueio, atualizar status do empréstimo, enviar notificação).

**Memória de trabalho inicial (WM0)**

| Fato na WM | Significado |
|---|---|
| `(em_aberto, Emprestimo1)` | empréstimo não devolvido |
| `(prazo, Emprestimo1, Data10)` | prazo de devolução |
| `(data_atual, Data15)` / `(antes, Data10, Data15)` | referência temporal |
| `(usuario_de, Emprestimo1, Ana)` | responsável |
| `(notificacao, Ana, pendente)` | notificação ainda não enviada |

**Regras de produção**

| Regra | Prioridade | IF (condições sobre WM) | THEN (ações) |
|---|---|---|---|
| RP1 | 1 | em_aberto(e) ∧ prazo(e,d1) ∧ data_atual(d2) ∧ antes(d1,d2) | ADD (atraso,e) |
| RP2 | 2 | atraso(e) ∧ usuario_de(e,u) | ADD (pendencia,u) |
| RP3 | 3 | pendencia(u) | ADD (bloqueio,u) |
| RP4 | 4 | atraso(e) | MODIFY (em_aberto,e) → (atrasado_reportado,e) |
| RP5 | 5 | bloqueio(u) ∧ notificacao(u,pendente) | REMOVE (notificacao,u,pendente); ADD (notificacao,u,enviada) |

**Ciclo reconhecer → resolver conflito → agir:** a cada ciclo monta-se o *conjunto de conflito* (todas as ativações cujo antecedente casa com a WM). **Estratégia de seleção:** menor número de prioridade (empate → ordem de declaração). **Refratariedade:** uma ativação (regra + ligação) já disparada não dispara de novo. **Critério de parada:** conjunto de conflito vazio (ou limite de segurança de 20 ciclos).

In [9]:
# ---- Motor de producao (didatico, independente da KB logica) ----
WM0 = {
    ('em_aberto', 'Emprestimo1'),
    ('prazo', 'Emprestimo1', 'Data10'),
    ('data_atual', 'Data15'),
    ('antes', 'Data10', 'Data15'),
    ('usuario_de', 'Emprestimo1', 'Ana'),
    ('notificacao', 'Ana', 'pendente'),
}

REGRAS_PROD = [
    ('RP1', 1,
     [('em_aberto', '?e'), ('prazo', '?e', '?d1'), ('data_atual', '?d2'), ('antes', '?d1', '?d2')],
     [('ADD', ('atraso', '?e'))]),
    ('RP2', 2,
     [('atraso', '?e'), ('usuario_de', '?e', '?u')],
     [('ADD', ('pendencia', '?u'))]),
    ('RP3', 3,
     [('pendencia', '?u')],
     [('ADD', ('bloqueio', '?u'))]),
    ('RP4', 4,
     [('atraso', '?e')],
     [('MODIFY', ('em_aberto', '?e'), ('atrasado_reportado', '?e'))]),
    ('RP5', 5,
     [('bloqueio', '?u'), ('notificacao', '?u', 'pendente')],
     [('REMOVE', ('notificacao', '?u', 'pendente')), ('ADD', ('notificacao', '?u', 'enviada'))]),
]

def casar(padrao, fato, lig):
    """Casa um padrao de condicao com um fato da WM, estendendo a ligacao ?var -> valor."""
    lig2 = dict(lig)
    for p, f in zip(padrao, fato):
        if p.startswith('?'):
            if p in lig2:
                if lig2[p] != f:
                    return None
            else:
                lig2[p] = f
        elif p != f:
            return None
    return lig2

def instanciar(padrao, lig):
    return tuple(lig.get(p, p) if isinstance(p, str) and p.startswith('?') else p for p in padrao)

def conjunto_de_conflito(wm, usadas):
    """Ativacoes aplicaveis (regra + ligacao ainda nao usada)."""
    ativas = []
    for nome, prio, conds, acoes in REGRAS_PROD:
        ligs = [dict()]
        for cond in conds:
            novas = []
            for lig in ligs:
                for f in wm:
                    if len(f) == len(cond):
                        l2 = casar(cond, f, lig)
                        if l2 is not None:
                            novas.append(l2)
            ligs = novas
            if not ligs:
                break
        for lig in ligs:
            chave = (nome, tuple(sorted(lig.items())))
            if chave not in usadas:
                ativas.append((prio, nome, lig, acoes, chave))
    ativas.sort(key=lambda a: (a[0], a[1]))  # estrategia: prioridade, depois nome
    return ativas

def executar_producao():
    """Executa o ciclo reconhecer-resolver-agir ate a parada; devolve (WM final, historico, n_ciclos)."""
    wm, usadas, historico = set(WM0), set(), []
    for ciclo in range(1, 21):  # limite de seguranca (evita disparo infinito)
        ativas = conjunto_de_conflito(wm, usadas)
        registro = {'ciclo': ciclo, 'conflito': [(a[0], a[1]) for a in ativas], 'selecionada': None, 'acoes': []}
        if not ativas:
            historico.append(registro)
            return wm, historico, ciclo - 1
        prio, nome, lig, acoes, chave = ativas[0]
        registro['selecionada'] = (nome, prio, {k: v for k, v in lig.items()})
        for acao in acoes:
            if acao[0] == 'ADD':
                f = instanciar(acao[1], lig); wm.add(f); registro['acoes'].append(('ADD', f))
            elif acao[0] == 'REMOVE':
                f = instanciar(acao[1], lig); wm.discard(f); registro['acoes'].append(('REMOVE', f))
            elif acao[0] == 'MODIFY':
                velho, novo = instanciar(acao[1], lig), instanciar(acao[2], lig)
                wm.discard(velho); wm.add(novo); registro['acoes'].append(('MODIFY', velho, novo))
        usadas.add(chave)
        historico.append(registro)
    return wm, historico, 20

wm_final, historico, n_ciclos = executar_producao()
print('EXECUCAO DO SISTEMA DE PRODUCAO (WM0 -> saturacao)\n')
for reg in historico:
    print(f"--- Ciclo {reg['ciclo']} ---")
    print('conjunto de conflito:', reg['conflito'] if reg['conflito'] else 'vazio -> criterio de parada')
    if reg['selecionada']:
        nome, prio, lig = reg['selecionada']
        print(f"selecionada: {nome} (prioridade {prio}) com {lig}")
        for acao in reg['acoes']:
            if acao[0] == 'MODIFY':
                print(f'  MODIFY {acao[1]} -> {acao[2]}')
            else:
                print(f'  {acao[0]} {acao[1]}')
print('\nWM final:')
for f in sorted(wm_final):
    print(' ', f)
print(f'\nCiclos executados: {n_ciclos}')

EXECUCAO DO SISTEMA DE PRODUCAO (WM0 -> saturacao)

--- Ciclo 1 ---
conjunto de conflito: [(1, 'RP1')]
selecionada: RP1 (prioridade 1) com {'?e': 'Emprestimo1', '?d1': 'Data10', '?d2': 'Data15'}
  ADD ('atraso', 'Emprestimo1')
--- Ciclo 2 ---
conjunto de conflito: [(2, 'RP2'), (4, 'RP4')]
selecionada: RP2 (prioridade 2) com {'?e': 'Emprestimo1', '?u': 'Ana'}
  ADD ('pendencia', 'Ana')
--- Ciclo 3 ---
conjunto de conflito: [(3, 'RP3'), (4, 'RP4')]
selecionada: RP3 (prioridade 3) com {'?u': 'Ana'}
  ADD ('bloqueio', 'Ana')
--- Ciclo 4 ---
conjunto de conflito: [(4, 'RP4'), (5, 'RP5')]
selecionada: RP4 (prioridade 4) com {'?e': 'Emprestimo1'}
  MODIFY ('em_aberto', 'Emprestimo1') -> ('atrasado_reportado', 'Emprestimo1')
--- Ciclo 5 ---
conjunto de conflito: [(5, 'RP5')]
selecionada: RP5 (prioridade 5) com {'?u': 'Ana'}
  REMOVE ('notificacao', 'Ana', 'pendente')
  ADD ('notificacao', 'Ana', 'enviada')
--- Ciclo 6 ---
conjunto de conflito: vazio -> criterio de parada

WM final:
  ('antes',

## 9. Atualização dinâmica da KB (TELL / RETRACT)

Removemos o fato explícito `Anterior(Data10, Data15)` (F13) e observamos o efeito sobre a cadeia `R2 → R3 → R4`:
- **Antes:** PC7 provado.
- **Depois do `retract`:** PC7 deixa de ser **derivável** — e dizemos exatamente isso: *deixou de ser derivável*, **não** que "se tornou falso" (mundo aberto, item R2 do checklist).
- **Depois de re-inserir (`tell`):** a derivação é restaurada.

O `retract` remove apenas a sentença explícita; nenhuma conclusão derivada é apagada nem reavaliada automaticamente — limitação declarada do protótipo.

In [10]:
kb_up = construir_kb()
print('PC7 antes do retract:        ', 'provado' if list(fol_bc_ask(kb_up, expr('BloqueadoParaEmprestimo(Ana)'))) else 'nao provado')

kb_up.retract(expr('Anterior(Data10, Data15)'))  # remove F13
print('PC7 apos retract de F13:     ', 'provado' if list(fol_bc_ask(kb_up, expr('BloqueadoParaEmprestimo(Ana)'))) else 'nao provado (deixou de ser derivavel)')
print('PC5 apos retract de F13:     ', 'provado' if list(fol_bc_ask(kb_up, expr('EmprestimoAtrasado(Emprestimo1)'))) else 'nao provado (deixou de ser derivavel)')

kb_up.tell(expr('Anterior(Data10, Data15)'))      # restaura F13
print('PC7 apos re-TELL de F13:     ', 'provado' if list(fol_bc_ask(kb_up, expr('BloqueadoParaEmprestimo(Ana)'))) else 'nao provado')

PC7 antes do retract:         provado
PC7 apos retract de F13:      nao provado (deixou de ser derivavel)
PC5 apos retract de F13:      nao provado (deixou de ser derivavel)
PC7 apos re-TELL de F13:      provado


## 10. Testes automatizados de regressão

Sete testes cobrindo os casos exigidos: resposta explícita, resposta inferida, consulta com variável, caso não provado, atualização dinâmica e execução do sistema de produção.

In [11]:
def teste(nome, obtido, esperado):
    ok = obtido == esperado
    print(f'[{"PASS" if ok else "FAIL"}] {nome}  | obtido={obtido} esperado={esperado}')
    return ok

kb_t = construir_kb()
resultados = []

# T1 - resposta explicita (retrieval ground); ask() devolve False quando nao ha prova, senao a substituicao
resultados.append(teste('T1 PC1 explicito: Usuario(Ana)', kb_t.ask(expr('Usuario(Ana)')) is not False, True))

# T2 - resposta inferida via R1, com substitucao da variavel
subs = list(fol_bc_ask(kb_t, expr('PegouEmprestado(x, LivroA)')))
resultados.append(teste('T2 PC4 inferido: PegouEmprestado(x, LivroA) => x=Ana',
                        valor(subs[0], 'x') if subs else None, 'Ana'))

# T3 - cadeia de 3 regras (R2+R3+R4)
resultados.append(teste('T3 PC7 cadeia: BloqueadoParaEmprestimo(Ana)',
                        bool(list(fol_bc_ask(kb_t, expr('BloqueadoParaEmprestimo(Ana)')))), True))

# T4 - caso nao provado (mundo aberto)
resultados.append(teste('T4 PC10 nao provado: BloqueadoParaEmprestimo(Bruno)',
                        bool(list(fol_bc_ask(kb_t, expr('BloqueadoParaEmprestimo(Bruno)')))), False))

# T5 - varredura com variavel
bloqueados = [valor(s, 'x') for s in fol_bc_ask(kb_t, expr('BloqueadoParaEmprestimo(x)'))]
resultados.append(teste('T5 PC8 varredura: {x | bloqueado}', set(bloqueados), {'Ana'}))

# T6 - atualizacao dinamica (retract + re-tell)
kb_t6 = construir_kb()
antes = bool(list(fol_bc_ask(kb_t6, expr('BloqueadoParaEmprestimo(Ana)'))))
kb_t6.retract(expr('Anterior(Data10, Data15)'))
depois = bool(list(fol_bc_ask(kb_t6, expr('BloqueadoParaEmprestimo(Ana)'))))
kb_t6.tell(expr('Anterior(Data10, Data15)'))
restaurado = bool(list(fol_bc_ask(kb_t6, expr('BloqueadoParaEmprestimo(Ana)'))))
resultados.append(teste('T6 retract/tell: efeito e restauracao', (antes, depois, restaurado), (True, False, True)))

# T7 - sistema de producao executa e altera a WM (ADD/MODIFY/REMOVE)
wm_t, hist_t, ciclos_t = executar_producao()
acoes = [a[0] for reg in hist_t for a in reg['acoes']]
resultados.append(teste('T7 producao: WM final + acoes ADD/REMOVE/MODIFY',
                        ({('bloqueio', 'Ana'), ('notificacao', 'Ana', 'enviada'), ('atrasado_reportado', 'Emprestimo1')} <= wm_t,
                         {'ADD', 'MODIFY', 'REMOVE'} <= set(acoes),
                         hist_t[-1]['conflito'] == []),
                        (True, True, True)))

print(f'\n{sum(resultados)}/{len(resultados)} testes passaram.')
assert all(resultados)

[PASS] T1 PC1 explicito: Usuario(Ana)  | obtido=True esperado=True
[PASS] T2 PC4 inferido: PegouEmprestado(x, LivroA) => x=Ana  | obtido=Ana esperado=Ana
[PASS] T3 PC7 cadeia: BloqueadoParaEmprestimo(Ana)  | obtido=True esperado=True
[PASS] T4 PC10 nao provado: BloqueadoParaEmprestimo(Bruno)  | obtido=False esperado=False
[PASS] T5 PC8 varredura: {x | bloqueado}  | obtido={'Ana'} esperado={'Ana'}
[PASS] T6 retract/tell: efeito e restauracao  | obtido=(True, False, True) esperado=(True, False, True)
[PASS] T7 producao: WM final + acoes ADD/REMOVE/MODIFY  | obtido=(True, True, True) esperado=(True, True, True)

7/7 testes passaram.


## 11. Limitações e fronteiras do modelo

1. **Unicidade de categoria não formalizada:** R5 garante que todo livro tem *alguma* categoria, mas não impede formalmente múltiplas categorias por livro (exigiria igualdade formal `∀l∀c1∀c2 (CategoriaDoLivro(l,c1) ∧ CategoriaDoLivro(l,c2) → c1 = c2)`).
2. **Temporalidade não aritmética:** não há cálculo de dias em atraso — a ordem temporal depende de fatos explícitos `Anterior(d1,d2)`.
3. **Escopo simplificado:** sem multas cumulativas, renovações ou devoluções parciais.
4. **Restrição do motor SLD:** o `FolKB` opera apenas com cláusulas definidas puras; a regra existencial R5 não pôde ser codificada executavelmente (ver itens PC11/PC12).
5. **`retract` não reavalia conclusões:** remove a sentença explícita; conclusões derivadas (ex.: materializadas por um forward chaining anterior) não são automaticamente podadas — por isso as demonstrações de retract usam KB limpa.
6. **Sistema de produção didático:** motor próprio (não é RETE/Prolog), com refratariedade e limite de segurança de ciclos para impedir disparo infinito.

## Referências

- BRACHMAN, R. J.; LEVESQUE, H. J. *Knowledge Representation and Reasoning*. Morgan Kaufmann, 2004 (caps. 1–7, conforme checklist).
- AIMA-Python (`aima3`): `logic.py` — estruturas `Expr`/`FolKB`/`KB` e rotinas `tell`, `ask`, `ask_generator`, `retract`, `unify`, `subst`, `fol_fc_ask`, `fol_bc_ask`.
- Guias da disciplina COMP0513 – Aulas 01 a 10.